<a href="https://colab.research.google.com/github/group6aiml-hub/NIDS_AInML/blob/main/notebooks/07_Neural_Network_Hubert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Template: load data

Copy this notebook (**File → Save a copy in Drive**), rename it to `07a_<your model>.ipynb`
and build your model below. See `src/data.py` for what `d` contains and the rules in the README.

#1. Setup og Load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q imbalanced-learn
!rm -rf /content/NIDS_AInML
!git clone -q https://github.com/group6aiml-hub/NIDS_AInML.git
import sys; sys.path.append('/content/NIDS_AInML/src')
from data import last_data

d = last_data()
print(d['X_tr'].shape, d['X_val'].shape)   # expected: (86192, 41) (21548, 41)
print('Classes:', list(d['le'].classes_))

print("Features: ", d['X_tr'])

Mounted at /content/drive
(86192, 41) (21548, 41)
Classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']
Features:               dur  proto service state  spkts  dpkts  sbytes  dbytes  \
44435   0.000002    udp       -   INT      2      0     104       0   
3461    0.001064    udp     dns   CON      2      2     146     178   
93673   0.000011    udp       -   INT      2      0     168       0   
102977  0.639900    tcp       -   FIN     10      8     564     354   
26050   0.005607    tcp       -   FIN     22     24    2054    2478   
...          ...    ...     ...   ...    ...    ...     ...     ...   
79743   0.407582    tcp    http   FIN     10      8     824    1480   
88710   0.000009  other       -   INT      2      0     200       0   
28364   0.001540    udp       -   CON      4      4     520     304   
104347  0.742956    tcp     ftp   FIN     14     12     798     682   
8369    0.037877    tcp       -  

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


processor = d['lag_preprocess']()

x_train = d['X_tr']
y_train = d['y_tr_mc']
x_val = d['X_val']
y_val = d['y_val_mc']

prediction_Pipeline = Pipeline([
    ('processor', processor),
    ('model', MLPClassifier(max_iter=200))
])

# we can use this for the comparison of different activation functions, etc
def Create_Pipeline(activation_func, Xtrain=x_train, Ytrain=y_train, preprocessor=processor):
  prediction_Pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(activation=activation_func))
  ])

  prediction_Pipeline.fit(Xtrain, Ytrain)

  def Predict(pipeline=prediction_Pipeline, Xtest=x_val, Ytest=y_val):
    prediction = pipeline.predict(Xtest)
    accuracy = pipeline.score(Xtest, Ytest)

    accuracy = prediction_Pipeline.score(x_val, y_val)
    percentage = accuracy_score(y_val, prediction) * 100

    print("Accurarcy of neural network: ", accuracy)
    print("Percentage: ", percentage,"%")

  Predict()

  return prediction_Pipeline


print("RELU: ")
RELU_MLP = Create_Pipeline('relu')

print("Tanh: ")
TanH_MLP = Create_Pipeline('tanh')

print("Sigmoid: ")
Sigmoid_MLP = Create_Pipeline('logistic')

print("Identity: ")
Identity_MLP = Create_Pipeline('identity')

# prediction_Pipeline.fit(x_train, y_train)
# predictions = prediction_Pipeline.predict(x_val)


RELU: 


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Accurarcy of neural network:  0.803972526452571
Percentage:  80.3972526452571 %
Tanh: 


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Accurarcy of neural network:  0.8060144793020234
Percentage:  80.60144793020234 %
Sigmoid: 


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Accurarcy of neural network:  0.8067105995916094
Percentage:  80.67105995916094 %
Identity: 
Accurarcy of neural network:  0.7594672359383702
Percentage:  75.94672359383702 %


### Comparison between different Optimizers: Adam, SGF(Stochastic Gradient Finder), LBFGS

Since ReLU provided the highest accuracy(on the first attempt) in the prior block, we'll select it as the activation function for all the new models


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


processor = d['lag_preprocess']()

x_train = d['X_tr']
y_train = d['y_tr_mc']
x_val = d['X_val']
y_val = d['y_val_mc']

prediction_Pipeline = Pipeline([
    ('processor', processor),
    ('model', MLPClassifier(max_iter=200))
])

def Create_Pipeline2(Solver, Xtrain=x_train, Ytrain=y_train, preprocessor=processor):
  prediction_Pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(solver=Solver, activation="relu"))
  ])

  prediction_Pipeline.fit(Xtrain, Ytrain)

  def Predict(pipeline=prediction_Pipeline, Xtest=x_val, Ytest=y_val):
    prediction = pipeline.predict(Xtest)
    accuracy = pipeline.score(Xtest, Ytest)

    accuracy = prediction_Pipeline.score(x_val, y_val)
    percentage = accuracy_score(y_val, prediction) * 100

    print("Accurarcy of neural network: ", accuracy)
    print("Percentage: ", percentage,"%")

  Predict()

  return prediction_Pipeline

print("Adam: ")
Adam_MLP = Create_Pipeline2('adam')

print("SGD: ")
SGD_MLP = Create_Pipeline2('sgd')

print("LBFGS: ")
LBFGS_MLP = Create_Pipeline2('lbfgs')

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Accurarcy of neural network:  0.8053183590124373
Percentage:  80.53183590124374 %


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


Accurarcy of neural network:  0.7905142008539076
Percentage:  79.05142008539076 %


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


Accurarcy of neural network:  0.7990996844254687
Percentage:  79.90996844254687 %


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['proto', 'service',
                                                   'state']),
                                                 ('skew',
                                                  Pipeline(steps=[('log',
                                                                   FunctionTransformer(feature_names_out='one-to-one',
                                                                                       func=<ufunc 'log1p'>)),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['dur', 'spkts', 'dpkts',
                                                   'sbytes', 'dbytes', 'rate',
                                                   'sload', 'dload', 'sloss',
                                                   'dloss...
                                                   'sjit', 'djit', 'tcprtt',
                                                   'synack', 'ackdat', 'smean',
                                                   'dmean', 'trans_depth',
                                                   'response_body_len',
                                                   'ct_srv_src', 'ct_state_ttl',
                                                   'ct_dst_ltm',
                                                   'ct_src_dport_ltm',
                                                   'ct_dst_sport_ltm',
                                                   'ct_dst_src_ltm',
                                                   'ct_ftp_cmd',
                                                   'ct_flw_http_mthd',
                                                   'ct_src_ltm', ...]),
                                                 ('num', StandardScaler(),
                                                  ['sttl', 'dttl', 'swin',
                                                   'stcpb', 'dtcpb',
                                                   'dwin'])])),
                ('model', MLPClassifier(solver='lbfgs'))])